# XAUUSD Ichimoku Reinforcement Learning Trading Bot

This notebook demonstrates how to use the `xauusd-ichi-rl` package (v2.1.0) along with the provided scripts to train an RL agent, backtest its performance, and export it as an MQL5 Expert Advisor.

### 1. Setup and Imports
We have already split the `XAUUSD.csv` into monthly files and moved the core scripts (`run_rl_v2.py`, `run_backtest.py`, `generate_mql5.py`) into the current directory.

In [ ]:
import os
from pathlib import Path
from run_rl_v2 import run_v2
from run_backtest import main as run_backtest
from generate_mql5 import generate_all
import json

# Set data directory to current directory
data_dir = Path.cwd()
print(f"Data directory: {data_dir}")

### 2. Training the PPO Agent
We will train the agent for a small number of timesteps (e.g., 10,000) for this test. In a real scenario, you would use 500,000 or more.

In [ ]:
results = run_v2(
    data_dir=data_dir,
    timesteps=10000,  # Reduced for testing
    sl=5.0,           # Stop Loss in $
    tp=3.0,           # Take Profit in $
    initial_balance=500.0,
    lot_size=0.05
)
print("Training and Test complete!")
print(f"Results: {results}")

### 3. Rule-based Backtest & Optimization
The `run_backtest` script allows you to optimize SL/TP parameters using a grid search.

In [ ]:
# Run optimizer for specific month
import sys
sys.argv = [
    "run_backtest.py",
    "--mode=optimize",
    "--year=2026",
    "--month=01"
]
run_backtest()
print("Optimization complete! Configs saved to models/top3_configs.json")

### 4. Export to MQL5
Once we have the best configurations, we can generate the `.mq5` files for MetaTrader 5.

In [ ]:
config_path = data_dir / "models" / "top3_configs.json"
if config_path.exists():
    with open(config_path, encoding="utf-8") as f:
        top3 = json.load(f)
    
    output_dir = data_dir / "mql5_output"
    generate_all(top3)
    print(f"✅ MQL5 Expert Advisors generated in {output_dir}")
else:
    print("❌ Optimization results not found. Please run the backtest cell first.")